# Falsification controls for recurring continuous-sign units

This notebook repeats the 5,000-clip pilot with exactly matched trained-contrastive, random-Transformer, and raw-coordinate/velocity tokenizers. All use K=100, 8-frame patches, the same clips and source split. Lexical inference uses concept-label permutations within signer, preserving each signer's inventory. Passing the previous exploratory gate is not enough: the trained tokenizer should outperform both controls before full-corpus distributional training.


In [ ]:
import torch
assert torch.cuda.is_available(), 'Choose a GPU runtime, then Run all again.'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import shutil, subprocess, sys
from pathlib import Path
if shutil.which('aria2c') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'aria2'], check=True)
REPO_URL = 'https://github.com/ss-sebastian/youtube-asl-skeleton-bert.git'
REPO_REF = 'agent/shape-aware-stgcn'
PROJECT = Path('/content/youtube-asl-skeleton-bert')
if PROJECT.exists(): shutil.rmtree(PROJECT)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(PROJECT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT), 'scikit-learn>=1.4'], check=True)


Upload the same `sign_unit_probe_inputs.zip` used by the first pilot. Shard 1 must be downloaded again because the earlier notebook correctly deleted the raw 37 GB archive after producing its verified result bundle.


In [ ]:
from google.colab import files
target = Path('/content/sign_unit_probe_inputs.zip')
if not target.exists():
    uploaded = files.upload()
    assert uploaded, 'No input bundle uploaded.'
    source = Path('/content') / next(iter(uploaded))
    if source != target: source.rename(target)
print('Input bundle ready:', f'{target.stat().st_size / 1024**2:.1f} MiB')


In [ ]:
import runpy
runpy.run_path(str(PROJECT / 'scripts/colab_unit_controls.py'), run_name='__main__')


In [ ]:
import pandas as pd
from IPython.display import display
comparison = pd.read_csv('/content/sign_unit_controls/results/control_comparison.csv')
display(comparison)
comparison.set_index('representation')[['lexical_top1', 'lexical_top5', 'lexical_mrr']].plot.bar(grid=True, title='Cross-signer lexical unit controls')
comparison.set_index('representation')[['split_half_ami', 'centroid_cosine']].plot.bar(grid=True, title='Cross-source codebook reproducibility')
